# 4. praktika: ebaluazioa eta iterazioa

**Helburua**: egokitutako eredua test multzo **osoan** ebaluatu, oinarri-lerroarekin alderatu, akatsak aztertu eta iterazio bat egin.

Behar da: 3. saioko egokitzailea (`txartelak-lora`) Drive-n.

## 1. Ingurunea

In [ ]:
# COLAB: liburutegiak instalatu (bertsio finkoak, unitate osoan berdinak)
!pip install -q transformers==5.22.2 peft==0.21.2 trl==1.17.1 datasets==5.2.0

In [ ]:
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# COLAB: Google Drive muntatu (emaitzak gordetzeko eta datuak irakurtzeko)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
# Aldatu zure Drive-ko karpetara. Bertan egon behar dute txartelak_train.jsonl eta txartelak_test.jsonl
KARPETA = "/content/drive/MyDrive/llm-egokitzapena"
os.makedirs(KARPETA, exist_ok=True)
TRAIN_FITX = f"{KARPETA}/txartelak_train.jsonl"
TEST_FITX = f"{KARPETA}/txartelak_test.jsonl"

# Fitxategiak Drive-n ez badaude, igo eskuz
if not os.path.exists(TRAIN_FITX):
    from google.colab import files  # COLAB
    igoak = files.upload()
    for izena in igoak:
        os.replace(izena, f"{KARPETA}/{izena}")
print(os.listdir(KARPETA))

In [ ]:
import json
def irakurri_jsonl(fitx):
    with open(fitx, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_lista = irakurri_jsonl(TRAIN_FITX)
test_lista = irakurri_jsonl(TEST_FITX)
SYSTEM = train_lista[0]["prompt"][0]["content"]   # sistema-mezua datuetatik (beti berdina!)
print(len(train_lista), len(test_lista))
print(SYSTEM)

## 2. Oinarria + egokitzailea kargatu

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODELOA = "Qwen/Qwen2.5-0.5B-Instruct"
EGOKITZAILEA = f"{KARPETA}/txartelak-lora"

tokenizer = AutoTokenizer.from_pretrained(MODELOA)
base = AutoModelForCausalLM.from_pretrained(MODELOA, dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, EGOKITZAILEA)
model.eval()

In [ ]:
import json

def ebaluatu_bat(sortua, zuzena):
    """sortua: ereduaren testua; zuzena: dict-a (erantzun zuzena)."""
    em = {"json_ok": False, "kategoria": False, "lehentasuna": False,
          "kokapena": False, "guztiz": False}
    try:
        s = json.loads(sortua)
    except json.JSONDecodeError:
        return em
    if not isinstance(s, dict):
        return em
    em["json_ok"] = True
    for eremua in ["kategoria", "lehentasuna", "kokapena"]:
        em[eremua] = s.get(eremua) == zuzena[eremua]
    em["guztiz"] = em["kategoria"] and em["lehentasuna"] and em["kokapena"]
    return em

In [ ]:
def sortu_sortaka(mezu_zerrendak, model, tamaina=16, max_new_tokens=60):
    """Mezu-zerrenda askorentzat erantzunak sortu, sortaka eta deterministikoki."""
    tokenizer.padding_side = "left"           # sortzeko: betegarria ezkerrean
    emaitzak = []
    for i in range(0, len(mezu_zerrendak), tamaina):
        testuak = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                   for m in mezu_zerrendak[i:i + tamaina]]
        sarrera = tokenizer(testuak, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            irteera = model.generate(**sarrera, max_new_tokens=max_new_tokens, do_sample=False,
                                     pad_token_id=tokenizer.pad_token_id)
        berriak = irteera[:, sarrera["input_ids"].shape[1]:]
        emaitzak += tokenizer.batch_decode(berriak, skip_special_tokens=True)
    return emaitzak

In [ ]:
import pandas as pd

def ebaluatu_multzoa(adibideak, sortuak):
    lerroak = []
    for adib, sortua in zip(adibideak, sortuak):
        zuzena = json.loads(adib["completion"][0]["content"])
        lerroak.append({"testua": adib["prompt"][-1]["content"], "sortua": sortua,
                        "zuzena": adib["completion"][0]["content"],
                        "mota": adib.get("mota", ""), **ebaluatu_bat(sortua, zuzena)})
    return pd.DataFrame(lerroak)

METRIKAK = ["json_ok", "kategoria", "lehentasuna", "kokapena", "guztiz"]

## 3. Hiru sistemak test multzo osoan

Egokitzailea desaktibatuta (`disable_adapter`) oinarrizko eredua dugu, kopia bat kargatu gabe.

In [ ]:
import random, time
random.seed(0)
ikus, adibideak = set(), []
for a in random.sample(train_lista, len(train_lista)):
    k = json.loads(a["completion"][0]["content"])["kategoria"]
    if k not in ikus:
        ikus.add(k); adibideak.append(a)
    if len(adibideak) == 4:
        break

def few_shot_mezuak(adib):
    m = [{"role": "system", "content": SYSTEM}]
    for a in adibideak:
        m += [a["prompt"][1], a["completion"][0]]
    return m + [adib["prompt"][1]]

emaitzak = {}
with model.disable_adapter():
    t0 = time.time(); s = sortu_sortaka([a["prompt"] for a in test_lista], model)
    emaitzak["oinarria_zero_shot"] = (ebaluatu_multzoa(test_lista, s), time.time() - t0)
    t0 = time.time(); s = sortu_sortaka([few_shot_mezuak(a) for a in test_lista], model, tamaina=8)
    emaitzak["oinarria_few_shot"] = (ebaluatu_multzoa(test_lista, s), time.time() - t0)
t0 = time.time(); s = sortu_sortaka([a["prompt"] for a in test_lista], model)
emaitzak["lora_r16_e3"] = (ebaluatu_multzoa(test_lista, s), time.time() - t0)

In [ ]:
taula = pd.DataFrame({izena: {**df[METRIKAK].mean().to_dict(), "segundoak": dt}
                      for izena, (df, dt) in emaitzak.items()}).T
taula.round(3)

## 4. Ikusiak vs berriak (orokortzea)

In [ ]:
df = emaitzak["lora_r16_e3"][0]
df.groupby("mota")[METRIKAK].mean().round(3)

In [ ]:
# Oinarriarekin ere bai: non hobetu da gehien?
pd.concat({izena: d.groupby("mota")["guztiz"].mean() for izena, (d, _) in emaitzak.items()}, axis=1).round(3)

## 5. Errore-analisia

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

def eremua(testua, gakoa):
    try:
        return str(json.loads(testua).get(gakoa))
    except Exception:
        return "JSON_TXARRA"

df["kat_zuzena"] = df["zuzena"].map(lambda t: eremua(t, "kategoria"))
df["kat_sortua"] = df["sortua"].map(lambda t: eremua(t, "kategoria"))
df["lehen_zuzena"] = df["zuzena"].map(lambda t: eremua(t, "lehentasuna"))
df["lehen_sortua"] = df["sortua"].map(lambda t: eremua(t, "lehentasuna"))

ConfusionMatrixDisplay.from_predictions(df["kat_zuzena"], df["kat_sortua"], xticks_rotation=45)
plt.title("Kategoria"); plt.show()
print(classification_report(df["lehen_zuzena"], df["lehen_sortua"], zero_division=0))

In [ ]:
okerrak = df[~df["guztiz"]]
print(len(okerrak), "akats\n")
for _, r in okerrak.head(15).iterrows():
    print(f"[{r['mota']}] {r['testua']}")
    print("   SORTUA:", r["sortua"])
    print("   ZUZENA:", r["zuzena"], "\n")

### Ariketa: sailkatu akatsak

Irakurri akatsak eta bete taula (zenbat akats mota bakoitzeko):

| Akats mota | Kopurua | Adibide bat | Irtenbide posiblea |
|---|---|---|---|
| JSON txarra / moztua | | | |
| Kategoria okerra | | | |
| Lehentasun okerra | | | |
| Kokapena deklinatuta (*gelan*) | | | |
| Kokapena asmatua / falta | | | |
| Beste bat | | | |

## 6. Ahanzte katastrofikoa

Galdera orokorrak, **sistema-mezurik gabe**. Egokitutako ereduak gogoratzen du nola erantzun?

In [ ]:
galdera_orokorrak = ["Zer da Python? Erantzun esaldi batean.", "Zenbat da 12 + 30?",
                     "Idatzi haiku bat udazkenari buruz.", "¿Cuál es la capital de Francia?",
                     "Itzuli ingelesera: Gaur eguraldi ona dago."]
mezuak = [[{"role": "user", "content": g}] for g in galdera_orokorrak]
with model.disable_adapter():
    oinarria = sortu_sortaka(mezuak, model, max_new_tokens=60)
egokitua = sortu_sortaka(mezuak, model, max_new_tokens=60)
for g, o, e in zip(galdera_orokorrak, oinarria, egokitua):
    print(">>>", g, "\n  OINARRIA:", o.strip()[:120], "\n  EGOKITUA:", e.strip()[:120], "\n")

> **Galdera**: zer gertatu da? Arazo bat da gure erabilera-kasurako? Noiz izango litzateke arazo?

## 7. Esperimentuak erregistratu

In [ ]:
import os
from datetime import datetime

def erregistratu(izena, konfig, metrikak, fitx=f"{KARPETA}/esperimentuak.csv"):
    lerroa = pd.DataFrame([{"data": datetime.now().isoformat(timespec="minutes"),
                            "izena": izena, **konfig, **metrikak}])
    if os.path.exists(fitx):                       # zutabe desberdinak izan ditzakete
        lerroa = pd.concat([pd.read_csv(fitx), lerroa], ignore_index=True)
    lerroa.to_csv(fitx, index=False)

for izena, (d, dt) in emaitzak.items():
    konfig = {"r": 16, "epoch": 3, "lr": 2e-4, "datuak": len(train_lista)} if izena.startswith("lora") else {}
    berriak = d[d["mota"] == "berria"]["guztiz"].mean()
    erregistratu(izena, konfig, {**d[METRIKAK].mean().round(3).to_dict(), "guztiz_berriak": round(berriak, 3)})

pd.read_csv(f"{KARPETA}/esperimentuak.csv")

## 8. Iteratu

Aukeratu aldaketa **bakar** bat errore-analisian oinarrituta, entrenatu berriz (3. koadernoa) eta ebaluatu hemen `EGOKITZAILEA` aldatuta. Ideiak:

- **Datuak**: `txartelak_train_garbia.jsonl` (2. saioa), `txartelak_gehigarriak.jsonl` gehitu, aniztasun gehiago `sortu_datuak.py`-n (txantiloi berriak, gela berriak).
- **Hiperparametroak**: epoch gehiago/gutxiago, `r`, `learning_rate`.
- **Eredua**: `Qwen/Qwen2.5-1.5B-Instruct`.

Erregistratu emaitza `erregistratu()`-rekin. Hobetu da `guztiz_berriak`?

In [ ]:
# Bi egokitzaile alderatzeko: kargatu bigarrena izen batekin eta aktibatu
# model.load_adapter(f"{KARPETA}/txartelak-lora-v2", adapter_name="v2")
# model.set_adapter("v2")
# s = sortu_sortaka([a["prompt"] for a in test_lista], model)
# df_v2 = ebaluatu_multzoa(test_lista, s); df_v2[METRIKAK].mean()